# 🖋️ HiEnWrite — Pembersihan Data & Eksperimen VGG19 vs ViT-B16 (Satu Notebook)
### Prediksi skor kepribadian Big Five (OCEAN) dari citra tulisan tangan — implementasi proposal V12

Notebook ini menggabungkan seluruh alur penelitian dalam **satu file**:

| Bagian | Isi | Proposal V12 |
|---|---|---|
| **A (A1–A11)** | Audit & pembersihan dataset, seleksi manual, duplikat, split 75:25 tingkat penulis, 5-Fold grouped, prapemrosesan 224×224 | Bagian 2, 3, 4.3 – 4.6 |
| **B (B1–B12)** | Model VGG19 & ViT-B16, 6 konfigurasi optimizer, 5-Fold CV, pemilihan konfigurasi, uji terisolasi, Tabel 8 (hasil evaluasi) & ΔRMSE | Bagian 4.7 – 4.10, 5, 6 |

> ℹ️ Proposal V12 **tidak memuat augmentasi data**, sehingga tidak ada augmentasi di notebook ini.

## ▶️ Cara menjalankan di Kaggle

1. Impor notebook ini dan tambahkan dataset HiEnWrite lewat **Add Input**. Folder dataset harus memuat `data/merged_data.csv` dan `images/`. Jika pencarian otomatis gagal, isi `DATA_ROOT` di A1 dengan path folder tersebut.
2. Pilih **Accelerator → GPU** di **Session options**. Untuk mengunduh bobot saat sesi berjalan, nyalakan **Internet** dan gunakan `ALLOW_WEIGHT_DOWNLOADS = True` di B1. Notebook mencari bobot di Input dan cache sebelum mengunduh.
3. Jalankan **Run All** dengan `SMOKE_TEST = True` di B1. Uji cepat memakai fold 0 dan maksimum 2 epoch per unit CV. Periksa ringkasan data di A7–A8 dan gunakan hasil mode ini untuk memeriksa alur.
4. Untuk eksperimen penuh, pastikan keputusan data di A1 sudah sesuai dengan penelitian Anda, ubah `SMOKE_TEST = False`, lalu pilih **Save & Run All**. Untuk melanjutkan sesi sebelumnya, tambahkan output sesi tersebut sebagai Input. Unit CV yang sudah selesai dengan konfigurasi yang cocok akan dilewati.

### Memakai bobot tanpa internet di Kaggle

Unduh ketiga berkas berikut di perangkat yang punya akses internet. Tambahkan sebagai satu dataset Kaggle, lalu pasang dataset bobot melalui **Add Input**.

| Lokasi dalam dataset bobot | Unduhan resmi |
|---|---|
| `vgg19-dcbb9e9d.pth` | [Bobot VGG19 ImageNet](https://download.pytorch.org/models/vgg19-dcbb9e9d.pth) |
| `vit-base-patch16-224/config.json` | [Konfigurasi ViT-B16](https://huggingface.co/google/vit-base-patch16-224/resolve/main/config.json?download=true) |
| `vit-base-patch16-224/model.safetensors` | [Bobot ViT-B16](https://huggingface.co/google/vit-base-patch16-224/resolve/main/model.safetensors?download=true) |

Pertahankan nama folder `vit-base-patch16-224` dan letakkan kedua berkas ViT di dalamnya. Atur `ALLOW_WEIGHT_DOWNLOADS = False` di B1. Jika ada beberapa salinan bobot di Input, isi `WEIGHTS_INPUT_DIR` dengan path dataset bobot yang dipilih. B7 akan mencetak lokasi bobot saat setiap arsitektur pertama kali dimuat.

> ⏱️ **Beban Bagian B:** 6 konfigurasi × 5 fold = 30 unit CV (+10 unit untuk kandidat batch size 32, default), tiap unit ≤ 50 epoch dan biasanya lebih singkat karena *early stopping*.

> ⚠️ **Titik yang perlu keputusan Anda:** (1) Openness baris Hindi konstan 24 (A3), (2) daftar seleksi manual belum dapat direproduksi penuh dari berkas ini (A5, A7).

## A1. Setup & Konfigurasi

In [ ]:
import os, glob, json, hashlib, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, ImageOps
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import KFold

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
LANCZOS = Image.Resampling.LANCZOS

# ====================== SESUAI PROPOSAL V12 ======================
SEED       = 42        # seed tetap: split & fold
IMG_SIZE   = 224       # Bagian 3: resize seragam 224 x 224
LABEL_MAX  = 40.0      # Bagian 3: skor OCEAN dibagi 40 (batas skala kuesioner, bukan statistik data)
TEST_RATIO = 0.25      # Bagian 4.4: 75 : 25 pada tingkat ID penulis
N_FOLDS    = 5         # Bagian 4.5
BLANK_STD_MIN = 5.0    # Bagian 2: citra kosong bila simpangan baku piksel < 5,0
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # Bagian 3: statistik ImageNet tetap
IMAGENET_STD  = [0.229, 0.224, 0.225]
RESIZE_MODE   = "squash"                # Bagian 3: resize langsung ke 224x224 ('pad' hanya untuk analisis sensitivitas)

# ====================== KEPUTUSAN & PARAMETER TEKNIS ======================
OPENNESS_FIX = "english_row"  # ⚠️ 'english_row' | 'none'  (lihat section A3)
NEAR_DUP_MAX_HAMMING = 8      # ambang dHash 16x16 (256 bit) untuk duplikat "hampir identik"
DHASH_N = 16

# ====================== SELEKSI MANUAL (Bagian 2) ======================
APPLY_MANUAL = True
# Koreksi orientasi: sudut PIL (positif = putar berlawanan arah jarum jam), diterapkan SETELAH koreksi EXIF.
# Hanya mengembalikan citra ke posisi tegak; tidak mengubah kemiringan huruf/garis dasar/spasi.
# Empat entri di bawah terverifikasi visual pada berkas ini (teks vertikal). Proposal menyebut 10 citra:
# sisanya (mis. terbalik 180°) tidak dapat dideteksi otomatis -> tambahkan dari daftar manual Anda.
MANUAL_ROTATE = {"38e.jpeg": 90, "38h.jpeg": 90, "195e.jpeg": -90, "195h.jpeg": -90}
# Penghapusan citra non-tulisan (mis. memuat wajah) atau salah-bahasa: {nama_berkas: alasan}. Berkas ini diperlakukan
# "tidak tersedia" (masuk aturan berkas_hilang seperti pada Gambar 2 proposal) sehingga penulisnya dikeluarkan beserta pasangannya.
MANUAL_REMOVE = {}            # contoh: {"123e.jpeg": "foto wajah"}

# ====================== ANGKA REFERENSI PROPOSAL V12 (Gambar 2 & Tabel 4) ======================
# (penulis dibuang, citra dibuang, citra bermasalah, citra sehat ikut terbuang)
PROPOSAL_REF = {"berkas_hilang": (31, 62, 44, 18), "tidak_terbaca": (1, 2, 1, 1), "citra_kosong": (1, 2, 1, 1),
                "duplikat_EN_HI_penulis_sama": (2, 4, 4, 0), "duplikat_antar_penulis": (46, 92, 78, 14)}
PROPOSAL_FINAL = dict(penulis=264, citra=528, train=198, test=66, fold_penulis=[40, 40, 40, 39, 39])

TRAITS = ["O", "C", "E", "A", "N"]
TRAIT_NAMES = {"O": "Openness", "C": "Conscientiousness", "E": "Extraversion", "A": "Agreeableness", "N": "Neuroticism"}

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
seed_everything()

In [ ]:
DATA_ROOT = None   # contoh manual: "/kaggle/input/nama-dataset-anda/HiEnWrite-Dataset-main"

def find_dataset_root():
    for base in ["/kaggle/input", ".", "/mnt/user-data", "/home/claude/data"]:
        hits = glob.glob(os.path.join(base, "**", "merged_data.csv"), recursive=True)
        if hits:
            return Path(hits[0]).parent.parent
    raise FileNotFoundError("merged_data.csv tidak ditemukan. Atur DATA_ROOT secara manual.")

ROOT     = Path(DATA_ROOT) if DATA_ROOT else find_dataset_root()
META_CSV = ROOT / "data" / "merged_data.csv"
IMG_DIR  = ROOT / "images"
OUT_DIR  = Path("/kaggle/working" if os.path.isdir("/kaggle/working") else ".") / "hienwrite_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset root :", ROOT)
print("Berkas gambar di folder:", len(list(IMG_DIR.glob("*"))), "| folder luaran:", OUT_DIR)

## A2. Memuat Metadata & Penghapusan Atribut Demografis
`writer_id` dan `lang` diturunkan dari nama berkas: `123e.jpeg` → penulis 123, Inggris; `123h.jpeg` → penulis 123, Hindi. Urutan kolom skor mengikuti **O-C-E-A-N**.

In [ ]:
raw = pd.read_csv(META_CSV).drop(columns=["Unnamed: 0"])
raw = raw.rename(columns={"image": "file", "Openness to Experience": "O", "Conscientiousness": "C",
                          "Extroversion": "E", "Agreeableness": "A", "Neuroticism": "N"})
parsed = raw["file"].str.extract(r"^(\d+)([eh])\.jpe?g$")
assert parsed.notna().all().all(), "Ada nama berkas yang tidak mengikuti pola <id><e|h>.jpeg"
raw["writer_id"] = parsed[0].astype(int)
raw["lang"] = parsed[1].map({"e": "EN", "h": "HI"})
raw = raw[["file", "writer_id", "lang"] + TRAITS + ["Age", "Gender"]]

print("Bentuk metadata:", raw.shape, "| nilai kosong:", int(raw.isna().sum().sum()),
      "| baris duplikat:", int(raw.duplicated().sum()))
print("Penulis unik:", raw.writer_id.nunique(), "| citra per penulis:", raw.groupby("writer_id").size().unique().tolist(),
      "| bahasa:", raw.lang.value_counts().to_dict())
assert raw[TRAITS].isna().sum().sum() == 0, "Ada nilai kosong pada skor OCEAN"
print("Rentang skor (skala kuesioner 0-40):"); display(raw[TRAITS].agg(["min", "max", "mean", "std"]).round(2))

meta = raw.drop(columns=["Age", "Gender"]).copy()        # demografi dibuang (Bagian 2)
print("Kolom setelah demografi dihapus:", list(meta.columns))

## A3. Audit Konsistensi Label (OCEAN)
Proposal menyatakan kedua citra seorang penulis berbagi label OCEAN yang sama. Asumsi ini diuji karena split dan fold tingkat penulis bergantung padanya.

In [ ]:
pair = meta.pivot(index="writer_id", columns="lang", values=TRAITS)
print("Jumlah penulis dengan skor EN != HI:", {TRAIT_NAMES[t]: int((pair[(t, "EN")] != pair[(t, "HI")]).sum()) for t in TRAITS})
display(meta.groupby("lang")["O"].agg(["count", "nunique", "min", "max", "mean", "std"]).round(2))
print("Nilai Openness pada baris Hindi:", meta.loc[meta.lang == "HI", "O"].value_counts().to_dict())

fig, axes = plt.subplots(1, 5, figsize=(18, 3.2))
for ax, t in zip(axes, TRAITS):
    sns.histplot(data=meta, x=t, hue="lang", bins=range(0, 42, 2), alpha=.55, ax=ax)
    ax.set_title(TRAIT_NAMES[t] + (" ⚠️ HI konstan" if t == "O" else ""))
plt.tight_layout(); plt.show()

**⚠️ Temuan:** empat dimensi (C, E, A, N) identik antara citra Inggris dan Hindi, tetapi **Openness pada seluruh baris Hindi konstan (24)**, sedangkan baris Inggris bervariasi. Nilai konstan ini hampir pasti artefak penggabungan data, bukan hasil kuesioner. Bila dibiarkan, target Openness citra Hindi tidak informatif dan model dapat "menebak" label hanya dari jenis aksara, sehingga RMSE tampak lebih baik secara semu, serta bertentangan dengan pernyataan proposal bahwa label kedua citra sama.

**Penanganan (`OPENNESS_FIX="english_row"`, default):** Openness penulis diambil dari baris Inggris dan disalin ke baris Hindi. Tidak ada skor lain yang diubah, dan tidak ada diskretisasi/normalisasi statistik. Ini mewujudkan prinsip "label sama pada kedua citra".

> 📝 Kalimat proposal *"skor OCEAN tidak diubah"* (Bagian 2) perlu disesuaikan bila Anda memakai default ini, mis.: *"skor OCEAN tidak diubah, kecuali Openness citra Hindi yang bernilai konstan (24) pada seluruh penulis dan diganti dengan skor Openness penulis yang sama dari citra Inggris."* Setel `OPENNESS_FIX="none"` bila ingin mengikuti kalimat proposal apa adanya (data mentah).

In [ ]:
meta["O_raw"] = meta["O"]
if OPENNESS_FIX == "english_row":
    en_open = meta.loc[meta.lang == "EN"].set_index("writer_id")["O"]
    meta["O"] = meta["writer_id"].map(en_open)
    print(f"Openness diseragamkan dari baris EN. Baris yang berubah: {int((meta.O != meta.O_raw).sum())}")
else:
    print("OPENNESS_FIX='none' -> skor mentah dipertahankan (Openness baris HI tetap konstan 24).")
n_uniq = meta.groupby("writer_id")[TRAITS].nunique().max()
print("Maks nilai unik per penulis per dimensi:", n_uniq.to_dict())
if OPENNESS_FIX == "english_row":
    assert (n_uniq == 1).all()

## A4. Audit Berkas Citra (seleksi otomatis)
Pemeriksaan per berkas (Bagian 2):
1. **Ketersediaan** berkas untuk tiap record metadata. Berkas pada `MANUAL_REMOVE` diperlakukan *tidak tersedia*.
2. **Keterbacaan**: dekode penuh; `LOAD_TRUNCATED_IMAGES` sengaja *tidak* diaktifkan agar citra terpotong tidak lolos diam-diam.
3. **Format/mode nyata** (beberapa `.jpeg` ternyata PNG/WEBP, sebagian RGBA → dikomposit ke latar **putih**, bukan hitam).
4. **Orientasi**: tag EXIF diterapkan, lalu koreksi manual `MANUAL_ROTATE`.
5. **Citra kosong**: simpangan baku piksel < `BLANK_STD_MIN` (5,0).
6. **Sidik jari** MD5 dan dHash untuk deteksi duplikat (section A6), serta fitur bantu kandidat seleksi manual (section A5).

In [ ]:
meta["manual_removed"] = meta["file"].isin(MANUAL_REMOVE) if APPLY_MANUAL else False
meta["file_exists"] = meta["file"].map(lambda f: (IMG_DIR / f).exists()) & ~meta["manual_removed"]
extra = sorted(set(os.listdir(IMG_DIR)) - set(meta.file))
print(f"Record metadata: {len(meta)} | berkas tersedia: {int(meta.file_exists.sum())} | TIDAK tersedia: {int((~meta.file_exists).sum())}"
      f" (di antaranya dihapus manual: {int(meta.manual_removed.sum())}) | berkas tanpa record: {len(extra)}")
miss = meta.loc[~meta.file_exists].groupby("writer_id").lang.apply(lambda s: "+".join(sorted(s)))
print("Penulis dengan berkas tidak tersedia -> kedua citra hilang:", int((miss == "EN+HI").sum()), "| salah satu:", int((miss != "EN+HI").sum()))

In [ ]:
def load_rgb(path):
    """Muat -> RGB: koreksi EXIF, alfa dikomposit ke latar PUTIH, lalu koreksi orientasi manual (kelipatan 90°, lossless)."""
    path = Path(path)
    with Image.open(path) as im:
        im.load()                                         # dekode penuh; error bila berkas terpotong
        im = ImageOps.exif_transpose(im)
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA"); bg = Image.new("RGBA", im.size, (255, 255, 255, 255)); im = Image.alpha_composite(bg, im)
        im = im.convert("RGB")
    ang = MANUAL_ROTATE.get(path.name, 0) if APPLY_MANUAL else 0
    return im.rotate(ang, expand=True) if ang else im

def _binarize(gray):
    return cv2.adaptiveThreshold(cv2.GaussianBlur(gray, (3, 3), 0), 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 41, 10)

def orientation_ratio(gray):
    """<1 berarti garis teks tampak vertikal (kandidat miring 90°); ~2-4 untuk teks horizontal."""
    b = _binarize(gray).astype(np.float32); h, w = b.shape
    b = b[int(h * .05):int(h * .95), int(w * .05):int(w * .95)]
    rp = np.convolve(b.sum(1), np.ones(7) / 7, "same"); cp = np.convolve(b.sum(0), np.ones(7) / 7, "same")
    return float((rp.std() / (rp.mean() + 1e-9)) / (cp.std() / (cp.mean() + 1e-9)))

def script_features(gray):
    """Fitur sederhana (HOG + proporsi garis horizontal panjang/shirorekha) untuk membedakan aksara Latin vs Devanagari."""
    from skimage.feature import hog
    b = _binarize(gray); out = []
    for sz in (128, 256):
        out.append(hog(cv2.resize(b, (sz, sz), interpolation=cv2.INTER_AREA), orientations=9,
                       pixels_per_cell=(sz // 8, sz // 8), cells_per_block=(2, 2), feature_vector=True))
    s = b.sum() + 1
    ex = [cv2.morphologyEx(b, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (L, 1))).sum() / s for L in (15, 25, 40, 60)]
    return np.r_[np.concatenate(out), ex]

def audit_one(fname):
    path = IMG_DIR / fname
    rec = dict(file=fname, readable=False, error="", fmt=None, mode=None, exif_orient=1, width=np.nan, height=np.nan,
               size_kb=round(path.stat().st_size / 1024, 1), gray_std=np.nan, orient_ratio=np.nan,
               md5=hashlib.md5(path.read_bytes()).hexdigest())
    bits = feat = None
    try:
        with Image.open(path) as im:
            rec["fmt"], rec["mode"] = im.format, im.mode
            rec["exif_orient"] = im.getexif().get(274, 1)
        rgb = load_rgb(path)
        rec["width"], rec["height"] = rgb.size
        g = rgb.convert("L")
        rec["gray_std"] = float(np.asarray(g.resize((256, 256), Image.Resampling.BILINEAR), dtype=np.float32).std())
        a = np.asarray(g.resize((DHASH_N + 1, DHASH_N), LANCZOS), dtype=np.int16)
        bits = (a[:, 1:] > a[:, :-1]).ravel()                              # dHash 16x16 = 256 bit
        small = rgb.copy(); small.thumbnail((640, 640)); gs = np.asarray(small.convert("L"))
        rec["orient_ratio"] = orientation_ratio(gs); feat = script_features(gs)
        rec["readable"] = True
    except Exception as e:
        rec["error"] = f"{type(e).__name__}: {e}"[:90]
    return rec, bits, feat

present = meta.loc[meta.file_exists, "file"].tolist()
results = [audit_one(f) for f in tqdm(present, desc="Audit citra")]
audit = pd.DataFrame([r for r, _, _ in results])
DH = {r["file"]: b for r, b, _ in results if b is not None}
SF = {r["file"]: f for r, _, f in results if f is not None}
audit["blank"] = audit.readable & (audit.gray_std < BLANK_STD_MIN)
audit.to_csv(OUT_DIR / "audit_images.csv", index=False)

print("Terbaca:", int(audit.readable.sum()), "| tidak terbaca:", int((~audit.readable).sum()))
display(audit.loc[~audit.readable, ["file", "error"]])
print(f"Citra kosong (std < {BLANK_STD_MIN}):"); display(audit.loc[audit.blank, ["file", "width", "height", "gray_std"]])
ok = audit[audit.readable]
print("Format asli:", ok.fmt.value_counts().to_dict(), "| mode:", ok["mode"].value_counts().to_dict())
print("Tag EXIF (1/0 = normal; 6/8 = diputar, dikoreksi):", ok.exif_orient.value_counts().to_dict())
display(ok[["width", "height", "size_kb", "gray_std"]].describe().round(1).T)

## A5. Seleksi Manual: Bantuan Kandidat
Proposal menetapkan seleksi manual untuk (1) citra yang tidak tegak (90°/180°) dan (2) citra yang memuat foto wajah. Dua hal ini memang tidak dapat diputuskan andal oleh kode, sehingga **keputusan akhir ada pada daftar manual Anda** (`MANUAL_ROTATE`, `MANUAL_REMOVE`). Sel di bawah hanya membantu menyaring kandidat:

- **Teks vertikal (kandidat miring 90°)**: rasio profil proyeksi baris/kolom. Pada setting default, empat citra (38e, 38h, 195e, 195h) sudah dikoreksi lewat `MANUAL_ROTATE`, sehingga daftar kandidat tersisa seharusnya kosong.
- **Aksara tidak sesuai nama berkas** (`e` seharusnya Inggris/Latin, `h` seharusnya Hindi/Devanagari): klasifier sederhana dengan validasi silang (akurasi ±92 %, jadi **hanya kandidat berkeyakinan tinggi yang ditampilkan dan harus diperiksa mata**). Ini temuan di luar proposal; lihat catatan di bawah.
- **Wajah** dan **terbalik 180°** tidak dideteksi otomatis (detektor wajah klasik menghasilkan banyak positif palsu pada foto kertas), sehingga periksa secara visual.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --- (a) kandidat teks vertikal ---
cand_rot = audit[audit.readable & (audit.orient_ratio < 0.7)].sort_values("orient_ratio")[["file", "orient_ratio", "width", "height"]]
print("Kandidat miring 90° yang BELUM dikoreksi (rasio < 0,7):", len(cand_rot)); display(cand_rot.round(2))

# --- (b) kandidat aksara tidak sesuai nama berkas ---
lf = sorted(SF); Xs = np.array([SF[f] for f in lf]); ys = np.array([f[-6] == "h" for f in lf]).astype(int)
clf = make_pipeline(StandardScaler(), PCA(60, random_state=0), LogisticRegression(C=0.01, max_iter=5000))
p_hi = cross_val_predict(clf, Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
lang = pd.DataFrame({"file": lf, "label_berkas": np.where(ys == 1, "HI", "EN"), "p_hindi": p_hi})
lang["curiga"] = ((lang.label_berkas == "EN") & (lang.p_hindi > .95)) | ((lang.label_berkas == "HI") & (lang.p_hindi < .05))
print(f"Akurasi klasifier aksara (validasi silang): {((lang.p_hindi > .5) == (ys == 1)).mean():.3f} -> hanya sebagai penyaring awal")
cand_lang = lang[lang.curiga].sort_values("p_hindi"); print("Kandidat aksara tidak sesuai (berkeyakinan tinggi):", len(cand_lang))
display(cand_lang[["file", "label_berkas", "p_hindi"]].round(2))

show = cand_lang.file.tolist()[:12]
if show:
    cols = 4; rows = (len(show) + cols - 1) // cols
    canvas = Image.new("RGB", (cols * 330, rows * 200), "white")
    for i, f in enumerate(show):
        im = load_rgb(IMG_DIR / f); im.thumbnail((320, 180)); canvas.paste(im, ((i % cols) * 330, (i // cols) * 200 + 14))
    fig, ax = plt.subplots(figsize=(15, 3.6 * rows)); ax.imshow(canvas); ax.axis("off")
    for i, f in enumerate(show):
        ax.text((i % cols) * 330 + 4, (i // cols) * 200 + 11, f, color="red", fontsize=9)
    plt.tight_layout(); plt.show()

> 📝 **Catatan temuan (di luar proposal):** pada berkas ini terdapat citra yang aksaranya tidak sesuai nama berkas, misalnya `88e` berisi Hindi, `104e`/`104h` tertukar, `117h` berisi Inggris. Pasangan yang hanya **tertukar** (104, 347) tetap valid untuk regresi karena label OCEAN benar; pasangan yang **sama-bahasa** (mis. 88, 117) melanggar rancangan "satu Inggris + satu Hindi" meski labelnya tidak salah. Di salinan proposal Anda, beberapa penulis tampaknya sudah dikeluarkan karena alasan ini (angka *berkas hilang* 31 penulis vs 25 pada berkas ini; penulis 88, 104, 117 ada pada daftar Gambar 2). Bila itu keputusan Anda, masukkan berkasnya ke `MANUAL_REMOVE` dan tuliskan sebagai bagian seleksi manual di naskah.

## A6. Duplikat Citra
Proposal (Bagian 2 dan 4.6) mensyaratkan deteksi duplikat **pada penulis yang sama maupun antar penulis**, lalu eksklusi seluruh penulis terkait **sebelum pembagian data**. Dua lapis: **MD5** (identik persis) dan **dHash 16×16** dengan jarak Hamming ≤ `NEAR_DUP_MAX_HAMMING` (menangkap berkas yang sama dengan warna/kompresi sedikit berbeda). Kelompok dibentuk dari komponen terhubung.

In [ ]:
valid = audit.loc[audit.readable & ~audit.blank, ["file", "md5"]].merge(meta[["file", "writer_id", "lang"]], on="file").reset_index(drop=True)
B = np.stack([DH[f] for f in valid.file]).astype(np.float32)
ham = B.shape[1] - (B @ B.T + (1 - B) @ (1 - B).T)
md5_code = pd.factorize(valid.md5)[0]
adj = (ham <= NEAR_DUP_MAX_HAMMING) | (md5_code[:, None] == md5_code[None, :])
_, valid["dup_cluster"] = connected_components(csr_matrix(adj), directed=False)

g = valid.groupby("dup_cluster")
valid["cluster_n_img"] = g.file.transform("size"); valid["cluster_n_writers"] = g.writer_id.transform("nunique")
valid["dup_same_writer"]  = (valid.cluster_n_img > 1) & (valid.cluster_n_writers == 1)   # EN identik HI pada penulis yang sama
valid["dup_cross_writer"] = valid.cluster_n_writers > 1                                    # citra sama pada beberapa ID penulis

print(f"Citra dalam kelompok duplikat: {int((valid.cluster_n_img > 1).sum())} | antar penulis: {int(valid.dup_cross_writer.sum())} citra / "
      f"{valid.loc[valid.dup_cross_writer, 'writer_id'].nunique()} penulis | sama penulis (EN=HI): "
      f"penulis {sorted(int(w) for w in valid.loc[valid.dup_same_writer, 'writer_id'].unique())}")
display(valid[valid.cluster_n_img > 1].groupby("dup_cluster")
        .agg(n_citra=("file", "size"), n_penulis=("writer_id", "nunique"), contoh=("file", lambda s: ", ".join(list(s)[:6])))
        .sort_values("n_citra", ascending=False).head(15))

h2 = ham.copy(); np.fill_diagonal(h2, 999); nn = h2.min(axis=1)
fig, ax = plt.subplots(figsize=(7, 3.4)); sns.histplot(nn, bins=50, ax=ax); ax.axvline(NEAR_DUP_MAX_HAMMING + .5, color="r", ls="--")
ax.set(xlabel="jarak Hamming ke tetangga terdekat (dari 256 bit)", title="Ambang duplikat (garis merah): puncak tajam di ~0, lalu lembah")
plt.tight_layout(); plt.show()

## A7. Aturan Eksklusi Tingkat Penulis & Himpunan Data Bersih (padanan Gambar 2)
Citra bermasalah dikeluarkan **beserta pasangannya** agar pasangan Inggris–Hindi tetap utuh. Lima aturan diterapkan berurutan seperti pada Gambar 2 proposal. Tabel menunjukkan, untuk tiap aturan, citra **bermasalah** versus citra **sehat yang ikut terbuang** karena pasangannya bermasalah, lalu dibandingkan dengan angka pada proposal.

In [ ]:
img = (meta.merge(audit[["file", "readable", "blank", "error"]], on="file", how="left")
           .merge(valid[["file", "dup_cluster", "dup_same_writer", "dup_cross_writer"]], on="file", how="left"))
for c in ["readable", "blank", "dup_same_writer", "dup_cross_writer"]:
    img[c] = img[c].fillna(False).astype(bool)
img["missing"] = ~img.file_exists
img["unreadable"] = img.file_exists & ~img.readable

RULES = [("berkas_hilang", "missing", "Berkas tercantum di CSV tetapi tidak ada di folder images/"),
         ("tidak_terbaca", "unreadable", "Berkas ada tetapi gagal didekode (rusak/terpotong)"),
         ("citra_kosong", "blank", f"Citra hampir polos (simpangan baku piksel < {BLANK_STD_MIN})"),
         ("duplikat_EN_HI_penulis_sama", "dup_same_writer", "Citra EN identik/hampir identik dengan citra HI milik penulis yang sama"),
         ("duplikat_antar_penulis", "dup_cross_writer", "Citra sama/hampir sama muncul pada ID penulis berbeda")]

remaining = pd.Series(True, index=sorted(img.writer_id.unique())); reason = pd.Series("", index=remaining.index)
rows = []
for name, col, desc in RULES:
    has = img.groupby("writer_id")[col].any()
    hit = has & remaining
    sub = img[img.writer_id.isin(hit[hit].index)]
    reason[hit] = name; remaining &= ~has
    rows.append(dict(aturan=name, keterangan=desc, penulis_dibuang=int(hit.sum()), citra_dibuang=len(sub),
                     citra_bermasalah=int(sub[col].sum()), citra_sehat_ikut_terbuang=len(sub) - int(sub[col].sum()),
                     penulis_tersisa=int(remaining.sum()), citra_tersisa=int(remaining.sum()) * 2,
                     id_penulis=", ".join(str(w) for w in sorted(hit[hit].index)[:12]) + (" ..." if hit.sum() > 12 else "")))
wf = pd.DataFrame(rows)
tot = dict(aturan="TOTAL", keterangan="", penulis_dibuang=wf.penulis_dibuang.sum(), citra_dibuang=wf.citra_dibuang.sum(),
           citra_bermasalah=wf.citra_bermasalah.sum(), citra_sehat_ikut_terbuang=wf.citra_sehat_ikut_terbuang.sum(),
           penulis_tersisa=wf.penulis_tersisa.iloc[-1], citra_tersisa=wf.citra_tersisa.iloc[-1], id_penulis="")
display(pd.concat([wf, pd.DataFrame([tot])], ignore_index=True).drop(columns=["keterangan"]))
wf.to_csv(OUT_DIR / "waterfall_eksklusi.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 3.8))
vals = [len(img)] + wf.citra_tersisa.tolist(); labs = ["Awal"] + wf.aturan.str.replace("_", "\n").tolist()
b = ax[0].bar(range(len(vals)), vals, color=["#4C72B0"] + ["#DD8452"] * (len(vals) - 2) + ["#55A868"])
ax[0].bar_label(b); ax[0].set_xticks(range(len(vals))); ax[0].set_xticklabels(labs, fontsize=7); ax[0].set_title("Citra tersisa setelah tiap aturan (waterfall)")
ax[1].barh(wf.aturan, wf.citra_bermasalah, color="#C44E52", label="citra bermasalah")
ax[1].barh(wf.aturan, wf.citra_sehat_ikut_terbuang, left=wf.citra_bermasalah, color="#8C8C8C", label="citra sehat (pasangan ikut terbuang)")
ax[1].invert_yaxis(); ax[1].legend(); ax[1].set_title("Citra yang dibuang menurut alasan"); plt.tight_layout(); plt.show()

clean = img[img.writer_id.isin(remaining[remaining].index)].copy()
cnt = clean.groupby("writer_id").agg(n=("file", "size"), n_en=("lang", lambda s: (s == "EN").sum()), n_hi=("lang", lambda s: (s == "HI").sum()))
assert (cnt.n == 2).all() and (cnt.n_en == 1).all() and (cnt.n_hi == 1).all(), "Ada penulis tanpa tepat 1 EN + 1 HI"
print(f"\nHimpunan bersih: {clean.writer_id.nunique()} penulis, {len(clean)} citra ({len(clean)/len(meta)*100:.1f}% dari {len(meta)} citra awal; "
      f"EN:HI = {(clean.lang=='EN').sum()}:{(clean.lang=='HI').sum()})")

excl = img[~img.writer_id.isin(remaining[remaining].index)].copy(); excl["alasan_penulis_dikeluarkan"] = excl.writer_id.map(reason)
excl[["file", "writer_id", "lang", "alasan_penulis_dikeluarkan", "missing", "unreadable", "blank", "dup_same_writer", "dup_cross_writer", "error"]
     ].to_csv(OUT_DIR / "exclusion_log.csv", index=False)

In [ ]:
# Perbandingan dengan angka pada proposal V12 (Gambar 2)
cmp = []
for _, r in wf.iterrows():
    ref = PROPOSAL_REF[r.aturan]; now = (r.penulis_dibuang, r.citra_dibuang, r.citra_bermasalah, r.citra_sehat_ikut_terbuang)
    cmp.append(dict(aturan=r.aturan, kode=str(now), proposal=str(ref), sama="✓" if tuple(map(int, now)) == ref else "≠"))
cmp.append(dict(aturan="penulis/citra tersisa", kode=f"{int(wf.penulis_tersisa.iloc[-1])} / {int(wf.citra_tersisa.iloc[-1])}",
                proposal=f"{PROPOSAL_FINAL['penulis']} / {PROPOSAL_FINAL['citra']}",
                sama="✓" if (int(wf.penulis_tersisa.iloc[-1]), int(wf.citra_tersisa.iloc[-1])) == (PROPOSAL_FINAL['penulis'], PROPOSAL_FINAL['citra']) else "≠"))
print("(penulis dibuang, citra dibuang, citra bermasalah, citra sehat ikut terbuang)"); display(pd.DataFrame(cmp))
if any(c["sama"] == "≠" for c in cmp):
    print("⚠️ Ada selisih terhadap proposal. Selisih pada 'berkas_hilang' berarti salinan data/daftar manual Anda berbeda dari berkas ini "
          "(mis. berkas yang Anda hapus manual). Isi MANUAL_REMOVE / MANUAL_ROTATE sesuai daftar Anda (section A1), atau perbarui angka pada naskah.")

## A8. Pembagian Data Tingkat Penulis (75 : 25) & 5-Fold Grouped CV
Sesuai Bagian 4.4 – 4.5:
- Unit pembagian adalah **ID penulis**, dipilih acak dengan **seed tetap**; kedua citra penulis selalu ikut subset yang sama. Jumlah penulis latih = `round(0,75 × N)` (mis. 264 × 0,75 = 198).
- Lima fold dibentuk **dari penulis data latih saja**, berbasis ID penulis (grouped k-fold); data uji tidak disentuh.
- Indeks fold dan daftar penulis uji disimpan sebagai **berkas tetap** (`fold_indices.csv`, `test_writer_ids.csv`) yang sama untuk VGG19 dan ViT-B16.

In [ ]:
clean["dup_cluster"] = clean["dup_cluster"].astype(int)
clean["group_id"] = clean["writer_id"]                       # unit split = ID penulis

units = np.array(sorted(clean.group_id.unique()))
perm = np.random.RandomState(SEED).permutation(units)
n_train = int(np.floor(len(units) * (1 - TEST_RATIO) + 0.5))   # pembulatan setengah-ke-atas
train_units, test_units = np.sort(perm[:n_train]), np.sort(perm[n_train:])

unit_fold = {}
for k, (_, va) in enumerate(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train_units)):
    for u in train_units[va]:
        unit_fold[u] = k

clean["split"] = np.where(clean.group_id.isin(test_units), "test", "train")
clean["fold"] = clean.group_id.map(unit_fold).fillna(-1).astype(int)       # -1 = data uji
for t in TRAITS:
    clean[f"{t}_s"] = clean[t] / LABEL_MAX                                  # pembagi tetap 40
assert clean[[f"{t}_s" for t in TRAITS]].min().min() >= 0 and clean[[f"{t}_s" for t in TRAITS]].max().max() <= 1

manifest = clean.sort_values(["writer_id", "lang"]).reset_index(drop=True)
manifest["img_idx"] = np.arange(len(manifest))
manifest = manifest[["img_idx", "file", "writer_id", "group_id", "lang", "split", "fold", "dup_cluster"] + TRAITS + [f"{t}_s" for t in TRAITS]]
manifest.to_csv(OUT_DIR / "split_manifest.csv", index=False)

# berkas tetap (catatan Tabel 5)
pd.DataFrame({"writer_id": test_units}).to_csv(OUT_DIR / "test_writer_ids.csv", index=False)
manifest[manifest.split == "train"].drop_duplicates("writer_id")[["writer_id", "fold"]].sort_values("writer_id").to_csv(OUT_DIR / "fold_indices.csv", index=False)

# ---- Tabel 4 ----
def summarize(df):
    return pd.Series({"Penulis": df.writer_id.nunique(), "Citra Inggris": (df.lang == "EN").sum(), "Citra Hindi": (df.lang == "HI").sum(), "Total citra": len(df)})
tab4 = manifest.groupby("split").apply(summarize).rename(index={"train": "Data latih (5-Fold CV)", "test": "Data uji terisolasi"})
tab4.loc["Total"] = tab4.sum(); tab4["Proporsi"] = (tab4["Total citra"] / tab4.loc["Total", "Total citra"] * 100).round(1).astype(str) + " %"
print("Tabel 4 (hasil kode):"); display(tab4)
fold_w = manifest[manifest.split == "train"].groupby("fold").writer_id.nunique().tolist()
fold_i = manifest[manifest.split == "train"].groupby("fold").size().tolist()
print("Penulis per fold:", fold_w, "| citra per fold:", fold_i)
print("Tabel 4 proposal   : latih", PROPOSAL_FINAL["train"], "penulis, uji", PROPOSAL_FINAL["test"], "penulis; fold", PROPOSAL_FINAL["fold_penulis"])
same = (tab4.loc["Data latih (5-Fold CV)", "Penulis"], tab4.loc["Data uji terisolasi", "Penulis"], sorted(fold_w, reverse=True)) == \
       (PROPOSAL_FINAL["train"], PROPOSAL_FINAL["test"], PROPOSAL_FINAL["fold_penulis"])
print("Kesesuaian dengan Tabel 4 proposal:", "✓ sama" if same else "≠ berbeda (mengikuti jumlah penulis bersih di section A7; perbarui Tabel 4 atau samakan seleksi manual)")

In [ ]:
w_level = manifest[manifest.lang == "EN"]
print("Rata-rata & simpangan baku skor (0-40) per split (informasi saja):"); display(w_level.groupby("split")[TRAITS].agg(["mean", "std"]).round(2))

## A9. Pemeriksaan Kebocoran Data (padanan Tabel 5)
Setiap mekanisme pencegahan pada Tabel 5 diuji sebagai **assertion**; sel berhenti dengan error bila ada yang dilanggar.

In [ ]:
tr = manifest[manifest.split == "train"]; te = manifest[manifest.split == "test"]
# Pembersihan: tak ada citra duplikat/hampir-duplikat tersisa (tiap kelompok duplikat = 1 citra)
assert manifest.dup_cluster.is_unique, "Masih ada citra duplikat pada data bersih"
# Split latih/uji: tak ada penulis yang sama di kedua subset; dua citra penulis selalu satu split & satu fold
assert set(tr.writer_id).isdisjoint(set(te.writer_id)), "Penulis ada di latih DAN uji"
per_w = manifest.groupby("writer_id").agg(n=("file", "size"), n_split=("split", "nunique"), n_fold=("fold", "nunique"))
assert (per_w.n == 2).all() and (per_w.n_split == 1).all() and (per_w.n_fold == 1).all()
# 5-Fold: fold saling lepas berbasis penulis, menutup seluruh data latih; uji tidak punya fold
assert sorted(tr.fold.unique()) == list(range(N_FOLDS)) and (te.fold == -1).all()
assert tr.groupby("writer_id").fold.nunique().max() == 1
# Berkas tetap konsisten dengan manifest dan tidak beririsan
tw = set(pd.read_csv(OUT_DIR / "test_writer_ids.csv").writer_id); fi = pd.read_csv(OUT_DIR / "fold_indices.csv")
assert tw == set(te.writer_id) and tw.isdisjoint(set(fi.writer_id)) and set(fi.writer_id) == set(tr.writer_id)
# Normalisasi/penskalaan: konstanta tetap (bukan statistik data)
assert IMAGENET_MEAN == [0.485, 0.456, 0.406] and IMAGENET_STD == [0.229, 0.224, 0.225] and LABEL_MAX == 40.0
print("✅ Semua pemeriksaan kebocoran lulus.")
print(f"   latih: {tr.writer_id.nunique()} penulis | uji: {te.writer_id.nunique()} penulis | fold latih (penulis): {fold_w}")

## A10. Prapemrosesan Citra → Cache `uint8` 224×224 (Bagian 3)
Untuk tiap citra pada manifest: koreksi orientasi → RGB (alfa ke latar putih) → *resize* 224×224 (LANCZOS, ber-*antialias*). Hasilnya disimpan sebagai array `uint8` `(N, 224, 224, 3)`. Skala 0–1 dan standardisasi ImageNet dilakukan pada Bagian B (section B4), setara dengan pelaksanaannya di setiap iterasi tetapi tanpa dekode berulang citra berukuran puluhan megapiksel.

> Catatan sesuai proposal: pengecilan citra dokumen utuh ke 224×224 mengurangi detail goresan mikro, dan *resize* langsung ke persegi mengubah rasio aspek (termasuk kemiringan huruf). Ini keterbatasan yang sudah dicatat pada Bagian 3; `RESIZE_MODE="pad"` tersedia bila ingin analisis sensitivitas.

In [ ]:
def resize_image(im, size=IMG_SIZE, mode=RESIZE_MODE):
    if mode == "squash":
        return im.resize((size, size), LANCZOS)
    if mode == "pad":
        scale = size / max(im.size); new = (max(1, round(im.width * scale)), max(1, round(im.height * scale)))
        canvas = Image.new("RGB", (size, size), (255, 255, 255)); canvas.paste(im.resize(new, LANCZOS), ((size - new[0]) // 2, (size - new[1]) // 2))
        return canvas
    raise ValueError(mode)

X = np.stack([np.asarray(resize_image(load_rgb(IMG_DIR / f)), dtype=np.uint8) for f in tqdm(manifest.file, desc="Resize 224x224")])
np.save(OUT_DIR / "images_224.npy", X)
assert X.shape == (len(manifest), IMG_SIZE, IMG_SIZE, 3) and X.dtype == np.uint8
print("images_224.npy:", X.shape, X.dtype, f"({X.nbytes/1e6:.0f} MB) | rentang piksel {X.min()}-{X.max()}")

In [ ]:
pick = pd.concat([manifest[manifest.lang == "EN"].sample(4, random_state=SEED), manifest[manifest.lang == "HI"].sample(4, random_state=SEED)])
fig, axes = plt.subplots(4, 4, figsize=(14, 10))
for k, (_, r) in enumerate(pick.iterrows()):
    row, col = (k // 4) * 2, k % 4; orig = load_rgb(IMG_DIR / r.file)
    axes[row, col].imshow(orig); axes[row, col].set_title(f"{r.file}  asli {orig.size[0]}x{orig.size[1]}", fontsize=9)
    axes[row + 1, col].imshow(X[r.img_idx]); axes[row + 1, col].set_title("224 x 224", fontsize=9)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## A11. Ringkasan Luaran

In [ ]:
summary = dict(
    config=dict(SEED=SEED, IMG_SIZE=IMG_SIZE, LABEL_MAX=LABEL_MAX, TEST_RATIO=TEST_RATIO, N_FOLDS=N_FOLDS, OPENNESS_FIX=OPENNESS_FIX,
                NEAR_DUP_MAX_HAMMING=NEAR_DUP_MAX_HAMMING, BLANK_STD_MIN=BLANK_STD_MIN, RESIZE_MODE=RESIZE_MODE,
                APPLY_MANUAL=APPLY_MANUAL, MANUAL_ROTATE=MANUAL_ROTATE, MANUAL_REMOVE=MANUAL_REMOVE,
                IMAGENET_MEAN=IMAGENET_MEAN, IMAGENET_STD=IMAGENET_STD, TRAITS=TRAITS, AUGMENTATION=False),
    counts=dict(rows_metadata=int(len(meta)), writers_metadata=int(meta.writer_id.nunique()), writers_clean=int(manifest.writer_id.nunique()),
                images_clean=int(len(manifest)), writers_train=int(tr.writer_id.nunique()), writers_test=int(te.writer_id.nunique()),
                fold_writers=fold_w))
(OUT_DIR / "config.json").write_text(json.dumps(summary, indent=2))
for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:24s} {p.stat().st_size/1e6:8.2f} MB")

### Lanjut ke Bagian B (eksperimen VGG19 vs ViT-B16)
Bagian B membaca langsung luaran Bagian A dari folder `hienwrite_clean/`; tidak ada yang perlu dipindahkan.

### Checklist
- [ ] Putuskan `OPENNESS_FIX` (section A3) dan sesuaikan kalimat "skor OCEAN tidak diubah" pada Bagian 2 proposal.
- [ ] Samakan `MANUAL_ROTATE` / `MANUAL_REMOVE` dengan daftar seleksi manual Anda (10 koreksi orientasi, 1 foto wajah, dan berkas lain yang Anda hapus), lalu pastikan tabel perbandingan di section A7 dan Tabel 4 di section A8 bertanda ✓.
- [ ] Bekukan keputusan di atas sebelum menjalankan Bagian B; jangan diubah setelah melihat hasil model.

*Dataset: Checker, Yadav & Katarya (2025), HiEnWrite, CC BY-NC 4.0, doi:10.1145/3756010.*

---
# BAGIAN B — Eksperimen Komparatif VGG19 vs ViT-B16
Mengimplementasikan Bagian **4.7 – 4.10**, **5**, dan **6** proposal V12.

| Proposal | Implementasi | Section |
|---|---|---|
| 4.7 / Tabel 6–7 | 2 arsitektur × 3 optimizer (Adam baseline, AdamW, SGD Nesterov); batch size 16 dengan kandidat 32 | B1, B4, B7 |
| 4.8 / tabel arsitektur | VGG19 & ViT-B16 pretrained ImageNet, head `Dense 256 → 128 → 5` | B4 |
| 4.9 / Persamaan (2) | Loss **RMSE** (ε = 10⁻⁸), end-to-end, early stopping (maks. 50 epoch, *patience* 10) | B5 |
| 4.5 / 4.8 | 5-Fold CV berbasis penulis, fold identik untuk kedua arsitektur | B7 |
| 4.8 | Konfigurasi terbaik per arsitektur dipilih dari **RMSE validasi rata-rata** (bukan uji) | B9 |
| 4.10 | Latih ulang pada seluruh data latih (epoch = rata-rata epoch terbaik lintas fold), uji **satu kali** | B10 |
| 6 / Tabel 8 (hasil evaluasi), Pers. (3)–(7) | RMSE keseluruhan & per dimensi, rata-rata ± SD antar-fold (ddof = 1), ΔRMSE = RMSE(VGG19) − RMSE(ViT-B16) | B8, B11 |

## Prinsip yang dijaga
- **Epoch, patience, optimizer, learning rate, dan fold tidak diubah dari proposal.** Beban dikurangi lewat *mixed precision*, cache citra, dan checkpoint, bukan dengan mengubah desain eksperimen.
- **Data uji tidak dipakai** untuk tuning, early stopping, atau pemilihan model; hanya disentuh sekali di B10, setelah seluruh CV selesai.
- Seed dan fold tetap; inisialisasi head sama untuk ketiga optimizer pada fold yang sama; **tanpa augmentasi**.

## B1. Setup & Konfigurasi

In [ ]:
import os, sys, json, time, glob, shutil, hashlib, random, gc, warnings, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

# ======================= KONTROL UTAMA =======================
SMOKE_TEST = True           # True = uji cepat (2 epoch, fold 0). Ubah ke False setelah lolos untuk eksperimen penuh
TIME_BUDGET_HOURS = None    # mis. 10.0 -> berhenti rapi sebelum batas sesi; lanjutkan di sesi berikutnya
ARTIFACT_DIR = None         # mis. "/kaggle/input/nama-input/hienwrite_clean" bila pencarian otomatis gagal

# ======================= SESUAI PROPOSAL (Tabel 6 & 7) =======================
SEED       = 42
N_FOLDS    = 5
MAX_EPOCHS = 50
PATIENCE   = 10             # early stopping pada RMSE validasi
EPS        = 1e-8           # Persamaan (2)
ARCHS      = ["vgg19", "vit_b16"]
OPTIMIZERS = ["adam", "adamw", "sgd"]          # adam = baseline
LR = {"vgg19":   {"adam": 1e-3, "adamw": 1e-3, "sgd": 1e-3},
      "vit_b16": {"adam": 1e-4, "adamw": 1e-4, "sgd": 1e-3}}
WD = {"adam": 0.0, "adamw": 0.01, "sgd": 1e-4}
BETAS, MOMENTUM = (0.9, 0.999), 0.9

# Tabel 7: batch size 16 (kandidat: 16 dan 32). Tahap 1 = matriks 6 konfigurasi pada bs=16.
# Tahap 2 (RUN_BS32=True) = kandidat bs=32 untuk optimizer terbaik tiap arsitektur; bs final dipilih dari RMSE validasi.
BATCH_SIZES_STAGE1 = [16]   # [16, 32] = matriks penuh 12 konfigurasi (2x lebih berat)
RUN_BS32 = True             # False = bs tetap 16 (nyatakan demikian di naskah)

# ======================= SUMBER BOBOT PRETRAINED =======================
WEIGHTS_INPUT_DIR = "/kaggle/input"
ALLOW_WEIGHT_DOWNLOADS = False      # False untuk memakai bobot dari Input/cache saja

# ======================= PILIHAN IMPLEMENTASI =======================
VIT_CKPT = "google/vit-base-patch16-224"   # ViT-B/16, pretrain ImageNet-21k, fine-tune ImageNet-1k
VIT_RENORM_TO_HALF = False  # False = ikuti proposal (ImageNet mean/std untuk kedua model).
                            # True  = ViT menerima mean=std=0.5 (normalisasi asli checkpoint) via konversi di dalam model.
USE_AMP = True              # mixed precision (hanya GPU); mempercepat tanpa mengubah desain eksperimen
SAVE_FINAL_WEIGHTS = True

if SMOKE_TEST:
    MAX_EPOCHS = 2
FOLDS = [0] if SMOKE_TEST else list(range(N_FOLDS))
TRAITS = ["O", "C", "E", "A", "N"]
TRAIT_NAMES = {"O": "Openness", "C": "Conscientiousness", "E": "Extraversion", "A": "Agreeableness", "N": "Neuroticism"}
LABEL_MAX = 40.0

# ---- lokasi artefak Bagian A & folder hasil ----
def find_artifact_dir():
    if ARTIFACT_DIR:
        return Path(ARTIFACT_DIR)
    for base in ["/kaggle/input", "/kaggle/working", "."]:
        hits = glob.glob(os.path.join(base, "**", "hienwrite_clean", "split_manifest.csv"), recursive=True)
        if hits:
            return Path(hits[0]).parent
    raise FileNotFoundError("split_manifest.csv tidak ditemukan. Tambahkan output Bagian A sebagai Input atau isi ARTIFACT_DIR.")

ART = OUT_DIR if ("OUT_DIR" in globals() and (OUT_DIR / "split_manifest.csv").exists()) else find_artifact_dir()   # Bagian A bila baru dijalankan; selain itu cari di Input
RUN_DIR = Path("/kaggle/working" if os.path.isdir("/kaggle/working") else ".") / ("hienwrite_runs_smoke" if SMOKE_TEST else "hienwrite_runs")
CV_DIR, FINAL_DIR, TAB_DIR = RUN_DIR / "cv", RUN_DIR / "final", RUN_DIR / "tables"
for d in (CV_DIR, FINAL_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Artefak data :", ART)
print("Folder hasil :", RUN_DIR, "| SMOKE_TEST =", SMOKE_TEST, "| MAX_EPOCHS =", MAX_EPOCHS, "| folds =", FOLDS)

In [ ]:
# ---- Lanjutkan dari sesi sebelumnya: salin checkpoint JSON dari output notebook ini yang dipasang sebagai Input ----
def import_previous_runs():
    if SMOKE_TEST:
        return
    copied = 0
    for src in glob.glob("/kaggle/input/**/" + RUN_DIR.name, recursive=True):
        src = Path(src)
        if src.resolve() == RUN_DIR.resolve():
            continue
        for f in src.rglob("*"):
            if f.is_file() and f.suffix != ".pt":                 # bobot tidak diperlukan untuk resume
                dst = RUN_DIR / f.relative_to(src)
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst); copied += 1
    print(f"Checkpoint dari sesi sebelumnya yang diimpor: {copied} berkas")
import_previous_runs()
print("Unit CV yang sudah ada:", len(list(CV_DIR.glob('*.json'))), "| model akhir yang sudah ada:", len(list(FINAL_DIR.glob('*.json'))))

## B2. Memuat Artefak Tahap Data
Manifest dan array citra berasal dari Bagian A. Dilakukan pemeriksaan konsistensi agar fold yang dipakai **identik** dengan yang sudah divalidasi bebas kebocoran.

In [ ]:
manifest = pd.read_csv(ART / "split_manifest.csv")
X = np.load(ART / "images_224.npy")                                   # (N,224,224,3) uint8
assert len(X) == len(manifest) and (manifest.img_idx.values == np.arange(len(manifest))).all()
assert X.dtype == np.uint8 and X.shape[1:] == (224, 224, 3)

Y = manifest[[f"{t}_s" for t in TRAITS]].to_numpy(np.float32)         # target skala 0-1 (urutan O,C,E,A,N)
is_train = (manifest.split == "train").to_numpy()
fold_arr = manifest.fold.to_numpy()
assert Y.min() >= 0 and Y.max() <= 1

# pemeriksaan ulang ringkas anti-kebocoran
tr, te = manifest[is_train], manifest[~is_train]
assert set(tr.writer_id).isdisjoint(set(te.writer_id))
assert tr.groupby("writer_id").fold.nunique().max() == 1 and (te.fold == -1).all()
assert sorted(tr.fold.unique()) == list(range(N_FOLDS))

# normalisasi: ikuti Bagian A bila config.json tersedia
cfg1 = {}
if (ART / "config.json").exists():
    cfg1 = json.loads((ART / "config.json").read_text()).get("config", {})
IMAGENET_MEAN = cfg1.get("IMAGENET_MEAN", [0.485, 0.456, 0.406])
IMAGENET_STD  = cfg1.get("IMAGENET_STD",  [0.229, 0.224, 0.225])
assert not cfg1.get("AUGMENTATION", False), "Proposal V12 tidak memakai augmentasi"
assert cfg1.get("LABEL_MAX", LABEL_MAX) == LABEL_MAX and cfg1.get("SEED", SEED) == SEED, "Konfigurasi berbeda dari Bagian A"
MANIFEST_MD5 = hashlib.md5(manifest[["file", "split", "fold"]].to_csv(index=False).encode()).hexdigest()   # hash isi pembagian data (stabil lintas sesi)

print(f"Citra: {len(manifest)} | latih: {int(is_train.sum())} ({tr.writer_id.nunique()} penulis) | uji: {int((~is_train).sum())} ({te.writer_id.nunique()} penulis)")
print("Ukuran fold (citra):", {int(k): int(v) for k, v in tr.groupby('fold').size().items()})
print("manifest md5:", MANIFEST_MD5[:10])

## B3. Baseline Referensi: Prediktor Rata-rata
Sebagai *sanity check* (**tambahan di luar proposal**), dihitung RMSE bila model hanya memprediksi **rata-rata skor data latih** untuk semua sampel. Rata-rata dihitung hanya dari data latih fold tersebut, sehingga tidak ada kebocoran. Model yang tidak lebih baik dari garis ini belum mempelajari apa pun dari tulisan tangan. Ini berguna untuk mendeteksi pelatihan yang "runtuh" ke nilai rata-rata, risiko yang nyata pada fine-tuning penuh VGG19 dengan Adam `lr=1e-3`.

In [ ]:
def rmse_from_err(err):
    """err: (n,5) selisih prediksi-aktual. Mengembalikan RMSE keseluruhan (Pers. 3) dan per dimensi (Pers. 4)."""
    mse_k = (err ** 2).mean(axis=0)
    return float(np.sqrt(mse_k.mean())), [float(v) for v in np.sqrt(mse_k)]

base_rows = []
for k in FOLDS:
    trm, vam = is_train & (fold_arr != k), is_train & (fold_arr == k)
    r, rk = rmse_from_err(Y[vam] - Y[trm].mean(0))
    base_rows.append(dict(fold=k, rmse=r, **{f"rmse_{t}": v for t, v in zip(TRAITS, rk)}))
base_cv = pd.DataFrame(base_rows)
BASE = dict(val_mean=base_cv.rmse.mean(), val_sd=base_cv.rmse.std(ddof=1) if len(base_cv) > 1 else float("nan"),
            val_trait=base_cv[[f"rmse_{t}" for t in TRAITS]].mean().values.tolist())
print(f"Baseline rata-rata | RMSE validasi (skala 0-1): {BASE['val_mean']:.4f} ± {BASE['val_sd']:.4f}"
      f"  (= {BASE['val_mean']*LABEL_MAX:.2f} pada skala 0-40)")
print("(Baseline untuk data uji baru dihitung di section B10, agar data uji tidak disentuh sebelum seluruh CV selesai.)")

## B4. Dataset, Model & Optimizer (PyTorch / Hugging Face)
- **Dataset:** citra 224×224 dari Bagian A → skala 0–1 → standardisasi ImageNet (Bagian 3). **Tanpa augmentasi** (tidak ada pada proposal V12), sehingga latih, validasi, dan uji diproses identik.
- **VGG19** (torchvision, bobot ImageNet): `features` konvolusi 3×3 → *Flatten* → `Dense 256 (ReLU) → Dense 128 (ReLU) → 5`.
- **ViT-B16** (Hugging Face `ViTModel`, bobot ImageNet-21k/1k): token `[CLS]` → `Dense 256 (ReLU) → Dense 128 (ReLU) → 5`.
- Seluruh bobot **dilatih end-to-end** (tanpa pembekuan backbone), sesuai Bagian 4.9.
- Optimizer sesuai Tabel 6: Adam (λ=0), AdamW (λ=0,01), SGD Nesterov (μ=0,9; λ=10⁻⁴).
- Seed diatur sebelum membangun model sehingga **head diinisialisasi sama** untuk ketiga optimizer pada fold yang sama.

- Bobot dicari di Kaggle Input, lalu di cache. Unduhan dari PyTorch atau Hugging Face dijalankan jika berkas lokal belum tersedia dan `ALLOW_WEIGHT_DOWNLOADS = True`.

> 🖥️ Sel B4–B5 membutuhkan PyTorch, torchvision, dan transformers. Gunakan GPU Kaggle untuk pelatihan.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torchvision
from torchvision.models import VGG19_Weights, vgg19
from functools import lru_cache
from urllib.error import URLError

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ON = bool(USE_AMP and device.type == "cuda")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)" if device.type == "cuda" else ""))
try:
    import transformers; print("transformers", transformers.__version__)
except ImportError:
    print("transformers belum terpasang -> jalankan: !pip install -q transformers")
if device.type != "cuda":
    print("⚠️ GPU tidak terdeteksi. Aktifkan Accelerator GPU; pelatihan di CPU akan sangat lambat.")

# ------------------- Dataset (tanpa augmentasi; Bagian 3 proposal) -------------------
# Setara dengan ToTensor (uint8 -> 0-1) + Normalize ImageNet, dihitung sekali agar pelatihan cepat dan deterministik.
_mean = np.array(IMAGENET_MEAN, np.float32).reshape(1, 3, 1, 1); _std = np.array(IMAGENET_STD, np.float32).reshape(1, 3, 1, 1)
X_t = torch.from_numpy(((X.transpose(0, 3, 1, 2).astype(np.float32) / 255.0) - _mean) / _std)
Y_t = torch.from_numpy(Y)
print("Tensor masukan:", tuple(X_t.shape), X_t.dtype, f"({X_t.numel() * 4 / 1e6:.0f} MB)")

def make_loader(idx, bs, shuffle, seed):
    g = torch.Generator(); g.manual_seed(seed); idx = torch.as_tensor(idx)
    return DataLoader(TensorDataset(X_t[idx], Y_t[idx]), batch_size=bs, shuffle=shuffle, generator=g, pin_memory=(device.type == "cuda"))

def cv_loaders(fold, bs):
    return (make_loader(np.flatnonzero(is_train & (fold_arr != fold)), bs, True, SEED + fold),
            make_loader(np.flatnonzero(is_train & (fold_arr == fold)), bs, False, SEED))

def final_loaders(bs):
    return (make_loader(np.flatnonzero(is_train), bs, True, SEED + 99), make_loader(np.flatnonzero(~is_train), bs, False, SEED))

# ------------------- Sumber bobot pretrained -------------------
VGG19_PRETRAINED = VGG19_Weights.IMAGENET1K_V1
VGG19_FILENAME = Path(VGG19_PRETRAINED.url).name
VGG19_SHA256 = "dcbb9e9dad569fff7a846263a77324fc34978fea2bfb039c012d710e1776ae44"
VIT_FILES = ("config.json", "model.safetensors")


def nonempty_file(path):
    return path.is_file() and path.stat().st_size > 0


def has_vit_weights(folder):
    return all(nonempty_file(folder / name) for name in VIT_FILES)


def find_weight_input(name, is_ready):
    root = Path(WEIGHTS_INPUT_DIR)
    matches = [path for path in sorted(root.rglob(name)) if is_ready(path)]
    if len(matches) > 1:
        raise ValueError(f"Ada beberapa salinan {name} di Input. Isi WEIGHTS_INPUT_DIR di B1 dengan path dataset bobot yang dipilih.")
    return matches[0] if matches else None


def weight_help(files):
    return (
        f"Tambahkan {files} lewat Add Input sesuai panduan di bagian atas notebook, "
        "atau aktifkan Internet di Session options dan set ALLOW_WEIGHT_DOWNLOADS = True di B1."
    )


@lru_cache(maxsize=1)
def vgg19_weights_file():
    path = find_weight_input(VGG19_FILENAME, nonempty_file)
    if path is None:
        path = Path(torch.hub.get_dir()) / "checkpoints" / VGG19_FILENAME
    if nonempty_file(path):
        digest = hashlib.sha256()
        with path.open("rb") as weights:
            for block in iter(lambda: weights.read(1024 * 1024), b""):
                digest.update(block)
        if digest.hexdigest() != VGG19_SHA256:
            raise ValueError(f"Checksum VGG19 tidak cocok untuk {path}. Ganti dengan berkas resmi dari panduan notebook.")
    else:
        if not ALLOW_WEIGHT_DOWNLOADS:
            raise FileNotFoundError(f"Bobot VGG19 belum tersedia. {weight_help(VGG19_FILENAME)}")
        print(f"Mengunduh bobot VGG19: {VGG19_PRETRAINED.url}")
        path.parent.mkdir(parents=True, exist_ok=True)
        try:
            torch.hub.download_url_to_file(VGG19_PRETRAINED.url, str(path), hash_prefix=VGG19_SHA256)
        except (URLError, TimeoutError, ConnectionError) as exc:
            raise RuntimeError(f"Unduhan bobot VGG19 gagal ({exc}). {weight_help(VGG19_FILENAME)}") from exc
    print(f"VGG19 | bobot: {path}")
    return path


@lru_cache(maxsize=1)
def vit_weights_folder():
    folder_name = VIT_CKPT.rsplit("/", 1)[-1]
    folder = find_weight_input(folder_name, has_vit_weights)
    if folder is None:
        # fallback: cari folder mana pun di Input yang langsung berisi config.json + model.safetensors
        root = Path(WEIGHTS_INPUT_DIR)
        cands = sorted({p.parent for p in root.rglob("model.safetensors") if has_vit_weights(p.parent)})
        if len(cands) > 1:
            raise ValueError(f"Ada beberapa folder bobot ViT di Input: {cands}. Isi WEIGHTS_INPUT_DIR di B1 dengan dataset yang dipilih.")
        folder = cands[0] if cands else None
    if folder is None:
        from huggingface_hub import snapshot_download, try_to_load_from_cache

        cached_weights = try_to_load_from_cache(VIT_CKPT, "model.safetensors")
        if isinstance(cached_weights, str) and has_vit_weights(Path(cached_weights).parent):
            folder = Path(cached_weights).parent

        if folder is None:
            files = f"folder {folder_name} berisi config.json dan model.safetensors"
            if not ALLOW_WEIGHT_DOWNLOADS:
                raise FileNotFoundError(f"Bobot ViT-B16 belum tersedia. {weight_help(files)}")
            print(f"Mengunduh bobot ViT-B16: {VIT_CKPT}")
            try:
                folder = Path(snapshot_download(repo_id=VIT_CKPT, allow_patterns=list(VIT_FILES)))
            except Exception as exc:
                raise RuntimeError(f"Unduhan bobot ViT-B16 gagal ({exc}). {weight_help(files)}") from exc
            if not has_vit_weights(folder):
                raise FileNotFoundError(f"Berkas ViT-B16 di {folder} belum lengkap. {weight_help(files)}")
    print(f"ViT-B16 | bobot: {folder}")
    return folder


# ------------------- Model -------------------
def _head(in_dim):
    return nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(inplace=True), nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Linear(128, 5))

class VGG19Regressor(nn.Module):
    def __init__(self):
        super().__init__()
        weights_path = vgg19_weights_file()
        base = vgg19(weights=None, init_weights=False)
        # SHA-256 sudah diverifikasi. Gunakan pemuatan checkpoint lama seperti torchvision.
        base.load_state_dict(torch.load(weights_path, map_location="cpu", weights_only=False))
        self.features, self.avgpool = base.features, base.avgpool         # 16 konv 3x3 + 5 max-pool; output 512x7x7
        self.head = nn.Sequential(nn.Flatten(), _head(512 * 7 * 7))
    def forward(self, x): return self.head(self.avgpool(self.features(x)))

class ViTRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        from transformers import ViTModel
        self.backbone = ViTModel.from_pretrained(
            str(vit_weights_folder()), add_pooling_layer=False, local_files_only=True, use_safetensors=True
        )
        self.head = _head(self.backbone.config.hidden_size)
        self.renorm = VIT_RENORM_TO_HALF
        self.register_buffer("m_in", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("s_in", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
    def forward(self, x):
        if self.renorm:                                    # ImageNet-normalized -> (x - 0.5) / 0.5
            x = (x * self.s_in + self.m_in - 0.5) / 0.5
        return self.head(self.backbone(pixel_values=x).last_hidden_state[:, 0])    # token [CLS]

def build_model(arch):
    return (VGG19Regressor() if arch == "vgg19" else ViTRegressor()).to(device)

def make_optimizer(model, arch, opt_name):
    lr, wd = LR[arch][opt_name], WD[opt_name]
    if opt_name == "adam":  return torch.optim.Adam(model.parameters(), lr=lr, betas=BETAS, eps=1e-8, weight_decay=0.0)
    if opt_name == "adamw": return torch.optim.AdamW(model.parameters(), lr=lr, betas=BETAS, eps=1e-8, weight_decay=wd)
    if opt_name == "sgd":   return torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, nesterov=True, weight_decay=wd)
    raise ValueError(opt_name)

def make_scaler():
    try:    return torch.amp.GradScaler("cuda", enabled=AMP_ON)
    except (AttributeError, TypeError): return torch.cuda.amp.GradScaler(enabled=AMP_ON)

## B5. Loss RMSE, Evaluasi & Loop Pelatihan
- **Loss (Persamaan 2):** `sqrt(MSE + ε)` pada satu *batch*, dirata-rata atas 5 dimensi.
- **RMSE evaluasi (Persamaan 3 dan 4):** dihitung dari **seluruh sampel** pada subset (bukan rata-rata RMSE per batch): per dimensi `√mean(err²)`, keseluruhan `√mean(MSE₅ dimensi)`.
- **Early stopping:** berhenti bila RMSE validasi keseluruhan tidak membaik selama `PATIENCE` epoch; `best_epoch` dicatat untuk pelatihan akhir.
- Tidak ada *learning-rate scheduler* karena tidak ada di proposal.

> ⚠️ *Catatan metodologis:* RMSE validasi yang dilaporkan adalah nilai pada epoch terbaik (yang dipilih oleh early stopping dari fold itu sendiri), sehingga sedikit optimistis. Itu sebabnya angka akhir yang jujur adalah RMSE uji terisolasi.

In [ ]:
def rmse_loss(pred, y):
    return torch.sqrt(F.mse_loss(pred.float(), y) + EPS)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    se = torch.zeros(5, device=device, dtype=torch.float64); n = 0; preds = []
    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ON):
            p = model(x)
        p = p.float()
        se += ((p - y) ** 2).sum(0).double(); n += len(y); preds.append(p.cpu())
    mse_k = se / n
    return float(mse_k.mean().sqrt()), [float(v) for v in mse_k.sqrt().cpu()], torch.cat(preds).numpy()

def train_epoch(model, loader, opt, scaler):
    model.train(); tot, n = 0.0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP_ON):
            pred = model(x)
        loss = rmse_loss(pred, y)
        if not torch.isfinite(loss):
            return float("nan")
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(y); n += len(y)
    return tot / n

def _cleanup(*objs):
    for o in objs: del o
    gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()

def train_unit(arch, opt_name, bs, fold, max_epochs=None, patience=None):
    """Satu unit 5-Fold CV: latih pada 4 fold, validasi pada 1 fold, early stopping. Mengembalikan dict hasil."""
    max_epochs = max_epochs or MAX_EPOCHS; patience = patience or PATIENCE
    set_seed(SEED + fold)                                   # head awal sama untuk semua optimizer pada fold ini
    model = build_model(arch); opt = make_optimizer(model, arch, opt_name); scaler = make_scaler()
    train_dl, val_dl = cv_loaders(fold, bs)
    t0 = time.time(); hist_tr, hist_va = [], []
    best_v, best_k, best_ep, bad, diverged = float("inf"), None, 0, 0, False
    for ep in range(1, max_epochs + 1):
        tl = train_epoch(model, train_dl, opt, scaler)
        if not np.isfinite(tl):
            diverged = True; break
        v, vk, _ = evaluate(model, val_dl)
        hist_tr.append(tl); hist_va.append(v)
        if v < best_v - 1e-6:
            best_v, best_k, best_ep, bad = v, vk, ep, 0
        else:
            bad += 1
            if bad >= patience: break
    if best_ep == 0:                                        # tidak pernah membaik / diverge pada epoch pertama
        best_v, best_k, best_ep = (hist_va[-1] if hist_va else float("nan")), ([float("nan")] * 5), len(hist_va)
    out = dict(best_epoch=best_ep, best_val_rmse=best_v, best_val_rmse_trait=best_k, epochs_run=len(hist_va),
               diverged=diverged, history=dict(train_loss=hist_tr, val_rmse=hist_va), seconds=time.time() - t0,
               n_train=len(train_dl.dataset), n_val=len(val_dl.dataset))
    _cleanup(model, opt, scaler, train_dl, val_dl)
    return out

def train_final(arch, opt_name, bs, epochs, save_path=None):
    """Pelatihan akhir pada SELURUH data latih selama `epochs` (tanpa validasi/early stopping), uji SATU kali di akhir."""
    set_seed(SEED + 99)
    model = build_model(arch); opt = make_optimizer(model, arch, opt_name); scaler = make_scaler()
    train_dl, test_dl = final_loaders(bs)
    t0 = time.time(); hist = []
    for ep in range(1, epochs + 1):
        hist.append(train_epoch(model, train_dl, opt, scaler))
    v, vk, preds = evaluate(model, test_dl)                 # <- satu-satunya evaluasi pada data uji
    if save_path is not None:
        torch.save(model.state_dict(), save_path)
    out = dict(test_rmse=v, test_rmse_trait=vk, epochs=epochs, train_loss=hist, seconds=time.time() - t0,
               n_train=len(train_dl.dataset), n_test=len(test_dl.dataset))
    _cleanup(model, opt, scaler, train_dl, test_dl)
    return out, preds

## B6. Pencatatan Hasil (Checkpoint & Resume)
Setiap unit (`arsitektur × optimizer × batch size × fold`) disimpan sebagai satu berkas JSON bertanda **signature** (hash seluruh hyperparameter, seed, md5 manifest). Saat dijalankan ulang:
- unit yang sudah ada **dan** signature-nya cocok → dilewati;
- signature berbeda (mis. Anda mengubah *learning rate*) → dihitung ulang, supaya hasil dari konfigurasi berbeda **tidak tercampur diam-diam**.

Granularitas checkpoint adalah per fold (menyimpan bobot + state optimizer tiap epoch untuk ViT/VGG19 terlalu besar untuk output Kaggle). Bila sesi mati di tengah satu fold, hanya fold itu yang diulang.

In [ ]:
def _sig(d):
    return hashlib.sha1(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:12]

def base_sig(arch, opt, bs):
    return dict(arch=arch, opt=opt, bs=bs, lr=LR[arch][opt], wd=WD[opt], betas=BETAS, momentum=MOMENTUM, eps=EPS,
                max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED, amp=AMP_ON if "AMP_ON" in globals() else USE_AMP,
                vit_ckpt=VIT_CKPT, vit_renorm=VIT_RENORM_TO_HALF,
                manifest=MANIFEST_MD5)

def _atomic_write(path, obj):
    tmp = Path(str(path) + ".tmp"); tmp.write_text(json.dumps(obj)); os.replace(tmp, path)

def cv_path(arch, opt, bs, fold): return CV_DIR / f"{arch}__{opt}__bs{bs}__fold{fold}.json"

def load_valid(path, sig):
    if not path.exists(): return None
    try: d = json.loads(path.read_text())
    except Exception: return None
    return d if d.get("signature") == sig else None

def cv_unit_done(arch, opt, bs, fold):
    return load_valid(cv_path(arch, opt, bs, fold), _sig(base_sig(arch, opt, bs))) is not None

def run_cv_unit(arch, opt, bs, fold):
    sig = _sig(base_sig(arch, opt, bs)); p = cv_path(arch, opt, bs, fold)
    res = train_unit(arch, opt, bs, fold)
    res.update(arch=arch, opt=opt, bs=bs, fold=fold, signature=sig)
    _atomic_write(p, res)
    return res

def load_cv_results():
    rows = []
    for arch in ARCHS:
        for opt in OPTIMIZERS:
            for bs in sorted({16, 32}):
                for fold in FOLDS:
                    d = load_valid(cv_path(arch, opt, bs, fold), _sig(base_sig(arch, opt, bs)))
                    if d is not None:
                        rows.append(dict(arch=arch, opt=opt, bs=bs, fold=fold, rmse=d["best_val_rmse"], best_epoch=d["best_epoch"],
                                         epochs_run=d["epochs_run"], diverged=d["diverged"], seconds=d["seconds"],
                                         **{f"rmse_{t}": v for t, v in zip(TRAITS, d["best_val_rmse_trait"])}))
    return pd.DataFrame(rows)

def summarize_cv(df):
    """Rata-rata ± SD (ddof=1, Persamaan 5-6) per konfigurasi; hanya konfigurasi dengan semua fold yang dianggap lengkap."""
    if df.empty: return df
    g = df.groupby(["arch", "opt", "bs"])
    s = g.agg(n_folds=("fold", "nunique"), mean_rmse=("rmse", "mean"), sd_rmse=("rmse", lambda v: v.std(ddof=1)),
              mean_best_epoch=("best_epoch", "mean"), diverged=("diverged", "sum"), minutes=("seconds", lambda v: v.sum() / 60),
              **{f"mean_{t}": (f"rmse_{t}", "mean") for t in TRAITS}).reset_index()
    s["complete"] = s.n_folds == len(FOLDS)
    return s

def fmt_ms(m, s): return f"{m:.4f} ± {s:.4f}" if np.isfinite(s) else f"{m:.4f}"

## B7. Menjalankan 5-Fold CV
Urutan eksekusi: *optimizer → arsitektur → fold*, sehingga VGG19 dan ViT-B16 untuk optimizer yang sama selesai berdampingan (perbandingan berpasangan sudah bisa dilihat lebih awal).

- **Tahap 1:** 6 konfigurasi pada batch size 16 (30 unit).
- **Tahap 2 (`RUN_BS32=True`, default):** optimizer terbaik tiap arsitektur diulang pada batch size 32 (kandidat pada Tabel 7), dengan biaya tambahan 10 unit; batch size final dipilih dari RMSE validasi rata-rata. Setel `BATCH_SIZES_STAGE1=[16, 32]` untuk matriks penuh 12 konfigurasi, atau `RUN_BS32=False` bila batch size dibakukan 16 (nyatakan di naskah).
- Tiap unit mencetak RMSE validasi terbaik, epoch terbaik, dan waktu. Bila `TIME_BUDGET_HOURS` terlampaui, eksekusi berhenti rapi dan `CV_COMPLETE=False`.

In [ ]:
T0 = time.time()

def out_of_time(est_unit_seconds):
    return TIME_BUDGET_HOURS is not None and (time.time() - T0 + est_unit_seconds) > TIME_BUDGET_HOURS * 3600

def run_cv(configs):
    est = 0.0
    todo = [(a, o, b, f) for a, o, b in configs for f in FOLDS if not cv_unit_done(a, o, b, f)]
    print(f"Unit tersisa: {len(todo)} dari {len(configs) * len(FOLDS)}")
    for n, (arch, opt, bs, fold) in enumerate(todo, 1):
        if out_of_time(est):
            print(f"⏸️ Batas waktu tercapai; {len(todo) - n + 1} unit tersisa. Lanjutkan di sesi berikutnya (unit selesai otomatis dilewati).")
            return False
        r = run_cv_unit(arch, opt, bs, fold); est = max(est, r["seconds"])
        flag = "  ⚠️ DIVERGE" if r["diverged"] else ""
        print(f"[{n:>2}/{len(todo)}] {arch:7s} {opt:5s} bs{bs:<2d} fold{fold} | val RMSE {r['best_val_rmse']:.4f} "
              f"(epoch {r['best_epoch']}/{r['epochs_run']}) | {r['seconds']/60:.1f} mnt{flag}")
    return True

configs1 = [(a, o, b) for o in OPTIMIZERS for a in ARCHS for b in BATCH_SIZES_STAGE1]
stage1_ok = run_cv(configs1)

stage2_ok = True
if RUN_BS32 and stage1_ok and 32 not in BATCH_SIZES_STAGE1:
    s1 = summarize_cv(load_cv_results()); s1 = s1[(s1.bs == 16) & s1.complete]
    configs2 = [(a, s1[s1.arch == a].sort_values("mean_rmse").iloc[0].opt, 32) for a in ARCHS]
    print("\nTahap 2 (bs=32) untuk optimizer terbaik per arsitektur:", configs2)
    stage2_ok = run_cv(configs2)

CV_COMPLETE = bool(stage1_ok and stage2_ok)
print("\nCV_COMPLETE =", CV_COMPLETE)

## B8. Rekap Hasil 5-Fold CV
RMSE pada skala 0–1 (kalikan 40 untuk skala kuesioner). Simpangan baku antar-fold memakai *ddof* = 1 (Persamaan 6).

In [ ]:
cv_df = load_cv_results()
cv_sum = summarize_cv(cv_df)
NAMES = {"vgg19": "VGG19", "vit_b16": "ViT-B16"}
OPT_NAMES = {"adam": "Adam (baseline)", "adamw": "AdamW", "sgd": "SGD Nesterov"}

def fold_table(arch, opt, bs):
    return cv_df[(cv_df.arch == arch) & (cv_df.opt == opt) & (cv_df.bs == bs)].set_index("fold").sort_index()

SEL = pd.DataFrame()
if cv_sum.empty:
    print("Belum ada hasil CV.")
else:
    comp = cv_sum[cv_sum.complete]
    # satu baris per (arsitektur, optimizer): batch size terpilih = RMSE validasi rata-rata terendah di antara kandidat yang selesai
    SEL = comp.sort_values("mean_rmse").groupby(["arch", "opt"], as_index=False).head(1)
    SEL = (SEL.assign(_a=SEL.arch.map({a: i for i, a in enumerate(ARCHS)}), _o=SEL.opt.map({o: i for i, o in enumerate(OPTIMIZERS)}))
              .sort_values(["_a", "_o"]).drop(columns=["_a", "_o"]).reset_index(drop=True))
    t9v = pd.DataFrame({"Arsitektur": SEL.arch.map(NAMES), "Optimizer": SEL.opt.map(OPT_NAMES),
                        "RMSE validasi (rata-rata ± SD)": [fmt_ms(m, s) for m, s in zip(SEL.mean_rmse, SEL.sd_rmse)],
                        "RMSE uji": "–"})
    print("Tabel 8 (bagian validasi; kolom RMSE uji diisi di B11). RMSE skala 0-1, SD antar-fold (ddof = 1):"); display(t9v)
    t9v.to_csv(TAB_DIR / "tabel8_validasi.csv", index=False)
    info = pd.DataFrame({"Arsitektur": SEL.arch.map(NAMES), "Optimizer": SEL.opt.map(OPT_NAMES), "BS terpilih": SEL.bs,
                         "RMSE validasi skala 0-40": [fmt_ms(m * LABEL_MAX, s * LABEL_MAX) for m, s in zip(SEL.mean_rmse, SEL.sd_rmse)],
                         "Epoch terbaik (rata2)": SEL.mean_best_epoch.round(1), "Diverge": SEL.diverged.astype(int)})
    print("\nInformasi pendukung (di luar Tabel 8): batch size terpilih, skala kuesioner, epoch terbaik:"); display(info)
    print(f"Referensi tambahan, baseline prediktor rata-rata: {fmt_ms(BASE['val_mean'], BASE['val_sd'])} (0-1) = {BASE['val_mean']*LABEL_MAX:.2f} (0-40)\n")

    detail = cv_sum.assign(Arsitektur=cv_sum.arch.map(NAMES), Optimizer=cv_sum.opt.map(OPT_NAMES),
                           **{"RMSE validasi": [fmt_ms(m, s) for m, s in zip(cv_sum.mean_rmse, cv_sum.sd_rmse)]})
    detail = detail.sort_values(["arch", "opt", "bs"])[["Arsitektur", "Optimizer", "bs", "n_folds", "RMSE validasi", "mean_best_epoch", "diverged"]]
    print("\nDetail seluruh konfigurasi yang dijalankan (termasuk kandidat batch size):"); display(detail.reset_index(drop=True))
    detail.to_csv(TAB_DIR / "cv_detail_konfigurasi.csv", index=False)

In [ ]:
if not cv_sum.empty:
    # ---- Tabel pendamping (format sama dengan Tabel 8): RMSE validasi per dimensi OCEAN, rata-rata ± SD antar-fold ----
    if len(SEL):
        prow = []
        for _, r in SEL.iterrows():
            d = fold_table(r.arch, r.opt, int(r.bs))
            row = {"Arsitektur": NAMES[r.arch], "Optimizer": OPT_NAMES[r.opt]}
            for t in TRAITS:
                row[TRAIT_NAMES[t]] = fmt_ms(d[f"rmse_{t}"].mean(), d[f"rmse_{t}"].std(ddof=1))
            prow.append(row)
        print("Tabel pendamping (validasi): RMSE per dimensi OCEAN, rata-rata ± SD antar-fold, skala 0-1:"); display(pd.DataFrame(prow))
        print("Referensi baseline rata-rata per dimensi:", {TRAIT_NAMES[t]: round(v, 4) for t, v in zip(TRAITS, BASE["val_trait"])})
        pd.DataFrame(prow).to_csv(TAB_DIR / "tabel8_pendamping_validasi_per_dimensi.csv", index=False)

        # ---- ΔRMSE (Persamaan 7) per fold untuk optimizer yang sama: RMSE(VGG19) - RMSE(ViT-B16); positif = ViT-B16 lebih kecil ----
        drow = []
        for opt in OPTIMIZERS:
            a, b = SEL[(SEL.arch == "vgg19") & (SEL.opt == opt)], SEL[(SEL.arch == "vit_b16") & (SEL.opt == opt)]
            if a.empty or b.empty: continue
            da, db = fold_table("vgg19", opt, int(a.bs.iloc[0])), fold_table("vit_b16", opt, int(b.bs.iloc[0]))
            common = da.index.intersection(db.index)
            row = {"Optimizer": OPT_NAMES[opt], "BS (VGG19/ViT)": f"{int(a.bs.iloc[0])}/{int(b.bs.iloc[0])}"}
            for col, nm in [("rmse", "Keseluruhan")] + [(f"rmse_{t}", TRAIT_NAMES[t]) for t in TRAITS]:
                dl = da.loc[common, col] - db.loc[common, col]
                row[nm] = f"{dl.mean():+.4f} ± {dl.std(ddof=1):.4f}" if len(dl) > 1 else f"{dl.mean():+.4f}"
            row["ViT lebih baik (fold)"] = f"{int(((da.loc[common, 'rmse'] - db.loc[common, 'rmse']) > 0).sum())}/{len(common)}"
            drow.append(row)
        print("ΔRMSE validasi = RMSE(VGG19) − RMSE(ViT-B16), rata-rata ± SD antar-fold (positif = ViT-B16 lebih kecil):"); display(pd.DataFrame(drow))
        pd.DataFrame(drow).to_csv(TAB_DIR / "delta_rmse_validasi.csv", index=False)

    fig, ax = plt.subplots(1, 2, figsize=(16, 4.2))
    s = cv_sum.sort_values(["arch", "opt", "bs"]).reset_index(drop=True)
    lab = [f"{NAMES[a]}\n{o}\nbs{b}" for a, o, b in zip(s.arch, s.opt, s.bs)]
    ax[0].bar(range(len(s)), s.mean_rmse, yerr=s.sd_rmse.fillna(0), capsize=4,
              color=["#4C72B0" if a == "vgg19" else "#DD8452" for a in s.arch])
    ax[0].axhline(BASE["val_mean"], color="r", ls="--", label="baseline rata-rata")
    ax[0].set_xticks(range(len(s))); ax[0].set_xticklabels(lab, fontsize=8); ax[0].set_ylabel("RMSE validasi (0-1)")
    ax[0].set_title("RMSE validasi rata-rata ± SD antar-fold"); ax[0].legend()
    ax[0].set_ylim(0, max(s.mean_rmse.max(), BASE["val_mean"]) * 1.25)

    best_now = s[s.complete].sort_values("mean_rmse").groupby("arch").head(1) if s.complete.any() else s.head(0)
    for _, r in best_now.iterrows():
        for f in FOLDS:
            d = load_valid(cv_path(r.arch, r.opt, int(r.bs), f), _sig(base_sig(r.arch, r.opt, int(r.bs))))
            if d: ax[1].plot(range(1, len(d["history"]["val_rmse"]) + 1), d["history"]["val_rmse"],
                             color="#4C72B0" if r.arch == "vgg19" else "#DD8452", alpha=.6,
                             label=f"{NAMES[r.arch]} · {r.opt}" if f == FOLDS[0] else None)
    ax[1].axhline(BASE["val_mean"], color="r", ls="--", lw=1)
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("RMSE validasi (0-1)"); ax[1].set_title("Kurva validasi (konfigurasi terbaik; satu garis per fold)")
    if len(best_now): ax[1].legend()
    plt.tight_layout(); (RUN_DIR / "figures").mkdir(exist_ok=True); plt.savefig(RUN_DIR / "figures" / "cv_overview.png", dpi=130); plt.show()

## B9. Memilih Konfigurasi Terbaik per Arsitektur
Kriteria (Bagian 4.8): **RMSE validasi rata-rata terendah** di antara konfigurasi yang seluruh fold-nya selesai. Data uji **tidak dilibatkan**. Jumlah epoch pelatihan akhir = **rata-rata `best_epoch` lintas fold** konfigurasi terpilih, dibulatkan (Bagian 4.10).

In [ ]:
BEST = {}
if not CV_COMPLETE:
    print("⏸️ CV belum lengkap -> pemilihan konfigurasi & uji akhir ditunda. Jalankan ulang notebook (unit selesai akan dilewati).")
else:
    comp = cv_sum[cv_sum.complete]
    for arch in ARCHS:
        r = comp[comp.arch == arch].sort_values("mean_rmse").iloc[0]
        BEST[arch] = dict(opt=r.opt, bs=int(r.bs), mean_rmse=float(r.mean_rmse), sd_rmse=float(r.sd_rmse),
                          final_epochs=max(1, int(round(r.mean_best_epoch))))
        print(f"{NAMES[arch]:8s} -> optimizer {OPT_NAMES[r.opt]:16s} bs={int(r.bs)} | RMSE val {fmt_ms(r.mean_rmse, r.sd_rmse)} "
              f"| epoch pelatihan akhir = {BEST[arch]['final_epochs']}")
    (RUN_DIR / "best_configs.json").write_text(json.dumps(BEST, indent=2))
    for arch in ARCHS:
        if BEST[arch]["mean_rmse"] >= BASE["val_mean"]:
            print(f"⚠️ {NAMES[arch]}: RMSE validasi tidak lebih baik dari baseline rata-rata. Periksa kurva/learning rate sebelum menafsirkan.")

## B10. Pelatihan Akhir & Uji Terisolasi (satu kali)
Untuk tiap arsitektur: latih ulang konfigurasi terpilih pada **seluruh data latih** selama jumlah epoch dari section B9 (tanpa validasi dan early stopping), lalu evaluasi **satu kali** pada data uji. Data uji tidak dipakai untuk memilih epoch, optimizer, maupun batch size. Hasil disimpan; menjalankan ulang sel ini tidak melatih ulang bila hasilnya sudah ada.

In [ ]:
FINAL = {}
if CV_COMPLETE:
    for arch in ARCHS:
        opt, bs, ep = BEST[arch]["opt"], BEST[arch]["bs"], BEST[arch]["final_epochs"]
        key = f"{arch}__{opt}__bs{bs}"; jp, pp = FINAL_DIR / f"{key}.json", FINAL_DIR / f"{key}__test_preds.npy"
        sig = _sig(dict(base_sig(arch, opt, bs), epochs=ep, kind="final"))
        d = load_valid(jp, sig)
        if d is not None and pp.exists():
            preds = np.load(pp); print(f"[{NAMES[arch]}] hasil akhir sudah ada -> dilewati")
        else:
            print(f"[{NAMES[arch]}] pelatihan akhir: {OPT_NAMES[opt]}, bs={bs}, {ep} epoch ...")
            d, preds = train_final(arch, opt, bs, ep, save_path=(FINAL_DIR / f"{key}.pt") if SAVE_FINAL_WEIGHTS else None)
            d.update(arch=arch, opt=opt, bs=bs, signature=sig); np.save(pp, preds); _atomic_write(jp, d)
        FINAL[arch] = dict(d, preds=preds)
        print(f"   RMSE uji (0-1) = {d['test_rmse']:.4f}  (= {d['test_rmse']*LABEL_MAX:.2f} pada skala 0-40) | {d['seconds']/60:.1f} mnt")

    # baseline prediktor rata-rata untuk data uji (rata-rata dari SELURUH data latih)
    BASE["test"], bt = rmse_from_err(Y[~is_train] - Y[is_train].mean(0))
    BASE["test_trait"] = dict(zip(TRAITS, bt))
    print(f"Baseline rata-rata, RMSE uji (0-1) = {BASE['test']:.4f}")

## B11. Hasil Akhir (padanan Tabel 8) & ΔRMSE
**ΔRMSE (Persamaan 7)** = `RMSE(VGG19) − RMSE(ViT-B16)` pada fold atau data uji yang sama; **nilai positif berarti ViT-B16 memiliki galat lebih kecil**. Tabel 8 di bawah memakai format persis rancangan proposal (Arsitektur, Optimizer, RMSE validasi rata-rata ± SD, RMSE uji; RMSE uji hanya untuk konfigurasi terbaik tiap arsitektur). Dua tabel pendamping berformat sama memuat nilai per dimensi OCEAN dan ΔRMSE.

In [ ]:
if CV_COMPLETE:
    def _is_best(r): return BEST[r.arch]["opt"] == r.opt and BEST[r.arch]["bs"] == int(r.bs)
    DIMS = [("Keseluruhan", "rmse", None)] + [(TRAIT_NAMES[t], f"rmse_{t}", i) for i, t in enumerate(TRAITS)]
    def _test(arch, idx): return FINAL[arch]["test_rmse"] if idx is None else FINAL[arch]["test_rmse_trait"][idx]

    # ================= Tabel 8: Hasil Evaluasi (format persis seperti rancangan proposal) =================
    tab8 = pd.DataFrame([{"Arsitektur": NAMES[r.arch], "Optimizer": OPT_NAMES[r.opt],
                          "RMSE validasi (rata-rata ± SD)": fmt_ms(r.mean_rmse, r.sd_rmse),
                          "RMSE uji": f"{FINAL[r.arch]['test_rmse']:.4f}" if _is_best(r) else "–"} for _, r in SEL.iterrows()])
    print("Tabel 8. Hasil Evaluasi (RMSE skala 0-1; RMSE uji hanya untuk konfigurasi terbaik tiap arsitektur):"); display(tab8)
    tab8.to_csv(TAB_DIR / "tabel8_hasil_evaluasi.csv", index=False)
    print("Catatan: kalikan 40 untuk skala kuesioner asli (0-40).")

    # ---- informasi pendukung (di luar Tabel 8) ----
    sup = pd.DataFrame([{"Arsitektur": NAMES[a], "Konfigurasi terbaik": OPT_NAMES[BEST[a]["opt"]], "BS": BEST[a]["bs"],
                         "Epoch pelatihan akhir": BEST[a]["final_epochs"], "RMSE uji (skala 0-40)": round(FINAL[a]["test_rmse"] * LABEL_MAX, 2)} for a in ARCHS])
    sup.loc[len(sup)] = ["Baseline rata-rata (referensi)", "-", "-", "-", round(BASE["test"] * LABEL_MAX, 2)]
    print("\nInformasi pendukung (di luar Tabel 8):"); display(sup); sup.to_csv(TAB_DIR / "informasi_pendukung.csv", index=False)

    # ================= Tabel pendamping 1: nilai per dimensi OCEAN (format sama dengan Tabel 8) =================
    data = []
    for _, r in SEL.iterrows():
        d = fold_table(r.arch, r.opt, int(r.bs)); rec = {}
        for nm, col, idx in DIMS:
            rec[(nm, "RMSE validasi (rata-rata ± SD)")] = fmt_ms(d[col].mean(), d[col].std(ddof=1))
            rec[(nm, "RMSE uji")] = f"{_test(r.arch, idx):.4f}" if _is_best(r) else "–"
        data.append(rec)
    comp_tab = pd.DataFrame(data, index=pd.MultiIndex.from_arrays([SEL.arch.map(NAMES), SEL.opt.map(OPT_NAMES)], names=["Arsitektur", "Optimizer"]))
    comp_tab.columns = pd.MultiIndex.from_tuples(comp_tab.columns)
    print("\nTabel pendamping A. RMSE per dimensi OCEAN (format sama dengan Tabel 8):"); display(comp_tab)
    comp_tab.to_csv(TAB_DIR / "tabel8_pendamping_per_dimensi.csv")

    # ================= Tabel pendamping 2: ΔRMSE = RMSE(VGG19) − RMSE(ViT-B16) (format sama) =================
    def _delta_val(opt_v, bs_v, opt_t, bs_t, col):
        da, db = fold_table("vgg19", opt_v, bs_v), fold_table("vit_b16", opt_t, bs_t); c = da.index.intersection(db.index)
        dl = da.loc[c, col] - db.loc[c, col]
        return f"{dl.mean():+.4f} ± {dl.std(ddof=1):.4f}" if len(dl) > 1 else f"{dl.mean():+.4f}"
    drows, didx = [], []
    for opt in OPTIMIZERS:                                   # pasangan optimizer yang sama, batch size terpilih masing-masing
        a, b = SEL[(SEL.arch == "vgg19") & (SEL.opt == opt)], SEL[(SEL.arch == "vit_b16") & (SEL.opt == opt)]
        if a.empty or b.empty: continue
        rec = {}
        for nm, col, idx in DIMS:
            rec[(nm, "ΔRMSE validasi (rata-rata ± SD)")] = _delta_val(opt, int(a.bs.iloc[0]), opt, int(b.bs.iloc[0]), col); rec[(nm, "ΔRMSE uji")] = "–"
        drows.append(rec); didx.append(OPT_NAMES[opt])
    rec = {}                                                 # konfigurasi terbaik tiap arsitektur: satu-satunya baris dengan nilai uji
    bv, bt = BEST["vgg19"], BEST["vit_b16"]
    for nm, col, idx in DIMS:
        rec[(nm, "ΔRMSE validasi (rata-rata ± SD)")] = _delta_val(bv["opt"], bv["bs"], bt["opt"], bt["bs"], col)
        rec[(nm, "ΔRMSE uji")] = f"{_test('vgg19', idx) - _test('vit_b16', idx):+.4f}"
    drows.append(rec); didx.append("Konfigurasi terbaik tiap arsitektur")
    delta_tab = pd.DataFrame(drows, index=pd.Index(didx, name="Pasangan (VGG19 − ViT-B16)")); delta_tab.columns = pd.MultiIndex.from_tuples(delta_tab.columns)
    print("\nTabel pendamping B. ΔRMSE = RMSE(VGG19) − RMSE(ViT-B16) (positif = ViT-B16 lebih kecil; uji hanya untuk konfigurasi terbaik):"); display(delta_tab)
    delta_tab.to_csv(TAB_DIR / "tabel8_pendamping_delta_rmse.csv")

    # ---- uji per dimensi dalam skala 0-1 dan 0-40, plus baseline referensi ----
    pt = pd.DataFrame({NAMES[a]: dict(zip(TRAITS, FINAL[a]["test_rmse_trait"]), Keseluruhan=FINAL[a]["test_rmse"]) for a in ARCHS}).T
    pt.loc["Baseline rata-rata (referensi)"] = dict(BASE["test_trait"], Keseluruhan=BASE["test"])
    pt = pt.rename(columns=TRAIT_NAMES)
    print("\nRMSE uji per dimensi, skala 0-1 (dan x40):"); display(pt.round(4)); display((pt * LABEL_MAX).round(2)); pt.to_csv(TAB_DIR / "uji_per_dimensi.csv")

    d_test = FINAL["vgg19"]["test_rmse"] - FINAL["vit_b16"]["test_rmse"]
    fv = pd.Series({f: json.loads(cv_path("vgg19", BEST["vgg19"]["opt"], BEST["vgg19"]["bs"], f).read_text())["best_val_rmse"] for f in FOLDS})
    fh = pd.Series({f: json.loads(cv_path("vit_b16", BEST["vit_b16"]["opt"], BEST["vit_b16"]["bs"], f).read_text())["best_val_rmse"] for f in FOLDS})
    d_fold = fv - fh
    print(f"\nΔRMSE uji (keseluruhan) = {d_test:+.4f} -> " + ("ViT-B16" if d_test > 0 else "VGG19") + " memiliki galat uji lebih kecil.")
    if len(d_fold) > 1:
        print(f"ΔRMSE validasi (konfigurasi terbaik) per fold: rata-rata {d_fold.mean():+.4f}, SD {d_fold.std(ddof=1):.4f}, ViT-B16 lebih baik pada {(d_fold > 0).sum()} dari {len(d_fold)} fold.")
        if abs(d_fold.mean()) < d_fold.std(ddof=1):
            print("ℹ️ Selisih rata-rata lebih kecil dari SD antar-fold: perbedaan antar-arsitektur belum dapat dipastikan (uji tunggal pada data uji kecil juga terbatas).")

In [ ]:
if CV_COMPLETE:
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1]})
    w = 0.38; xs = np.arange(len(TRAITS) + 1)
    for i, a in enumerate(ARCHS):
        vals = FINAL[a]["test_rmse_trait"] + [FINAL[a]["test_rmse"]]
        ax[0].bar(xs + (i - .5) * w, vals, w, label=NAMES[a], color="#4C72B0" if a == "vgg19" else "#DD8452")
    ax[0].plot(xs, list(BASE["test_trait"].values()) + [BASE["test"]], "r--o", ms=4, label="baseline rata-rata")
    ax[0].set_xticks(xs); ax[0].set_xticklabels([TRAIT_NAMES[t][:5] + "." for t in TRAITS] + ["Total"])
    ax[0].set_ylabel("RMSE uji (0-1)"); ax[0].set_title("RMSE uji per dimensi"); ax[0].legend()

    Yte = Y[~is_train]
    for a, c in zip(ARCHS, ["#4C72B0", "#DD8452"]):
        ax[1].scatter(Yte.ravel() * LABEL_MAX, FINAL[a]["preds"].ravel() * LABEL_MAX, s=7, alpha=.35, color=c, label=NAMES[a])
    ax[1].plot([0, LABEL_MAX], [0, LABEL_MAX], "k--", lw=1); ax[1].set_xlabel("skor aktual (0-40)"); ax[1].set_ylabel("prediksi (0-40)")
    ax[1].set_title("Prediksi vs aktual (seluruh dimensi, data uji)"); ax[1].legend()
    plt.tight_layout(); plt.savefig(RUN_DIR / "figures" / "test_overview.png", dpi=130); plt.show()

    manifest.loc[~is_train, ["file", "writer_id", "lang"]].assign(
        **{f"true_{t}": Y[~is_train][:, i] for i, t in enumerate(TRAITS)},
        **{f"{a}_pred_{t}": FINAL[a]["preds"][:, i] for a in ARCHS for i, t in enumerate(TRAITS)}
    ).to_csv(TAB_DIR / "test_predictions.csv", index=False)

meta_run = dict(SMOKE_TEST=SMOKE_TEST, MAX_EPOCHS=MAX_EPOCHS, PATIENCE=PATIENCE, SEED=SEED, FOLDS=FOLDS, USE_AMP=USE_AMP,
                VIT_CKPT=VIT_CKPT, VIT_RENORM_TO_HALF=VIT_RENORM_TO_HALF, LR=LR, WD=WD, MANIFEST_MD5=MANIFEST_MD5,
                CV_COMPLETE=CV_COMPLETE, python=platform.python_version())
(RUN_DIR / "run_meta.json").write_text(json.dumps(meta_run, indent=2))
print("Berkas hasil:", sorted(p.name for p in RUN_DIR.iterdir()))